In [ ]:
# Install dependencies once in your terminal:
# pip install -r requirements.txt


ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_

In [ ]:
from pathlib import Path
import os

repo_root = Path.cwd()
DATA_DIR = repo_root / "Dataset"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSON_FILE = DATA_DIR / "ai_papers_raw.json"
CSV_FILE = DATA_DIR / "ai_papers_clean.csv"

print("Repository root:", repo_root)
print("Dataset folder:", DATA_DIR)
print("Raw JSON file:", RAW_JSON_FILE)
print("Clean CSV file:", CSV_FILE)


In [ ]:
import json
import time
import os
import pandas as pd
import pyalex
from pyalex import Works


# ============================================================
# 1. CONFIG
# ============================================================

pyalex.config.email = os.getenv("OPENALEX_EMAIL", "your.email@example.com")
if pyalex.config.email == "your.email@example.com":
    raise RuntimeError("Set OPENALEX_EMAIL or update pyalex.config.email before running this notebook.")

MAX_RECORDS = 10000
PER_PAGE = 200
CURRENT_YEAR = 2026

# RAW_JSON_FILE and CSV_FILE are defined in the previous cell to keep outputs under Dataset/


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def safe_get(d, keys, default=None):
    cur = d
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def join_unique(values):
    values = safe_list(values)
    cleaned = [str(v) for v in values if v not in [None, ""]]
    return "; ".join(sorted(set(cleaned)))


# ============================================================
# 3. DEFINE CLEANER QUERY
# ============================================================

query = (
    Works()
    .search("artificial intelligence")
    .filter(
        has_doi=True,
        from_publication_date="2000-01-01",
        to_publication_date="2026-12-31",
        is_retracted=False,
        is_paratext=False,
    )
    .sort(cited_by_count="desc")
)

# Optional: keep only these after crawling
ALLOWED_TYPES = {
    "article",
    "preprint",
    "book-chapter",
    "proceedings-article",
}


# ============================================================
# 4. FETCH DATA
# ============================================================

all_papers = []

print("Starting data download from OpenAlex...")

for page in query.paginate(per_page=PER_PAGE):
    for paper in page:
        if paper.get("type") in ALLOWED_TYPES:
            all_papers.append(paper)

        if len(all_papers) >= MAX_RECORDS:
            break

    print(f"Downloaded usable papers: {len(all_papers)} / {MAX_RECORDS}")

    if len(all_papers) >= MAX_RECORDS:
        break

    time.sleep(0.2)

all_papers = all_papers[:MAX_RECORDS]

print(f"Successfully retrieved {len(all_papers)} papers.")


# ============================================================
# 5. SAVE RAW JSON
# ============================================================

with open(RAW_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(all_papers, f, ensure_ascii=False, indent=2)

print(f"Saved raw data to {RAW_JSON_FILE}")


# ============================================================
# 6. FLATTEN TO ONE CLEAN CSV
# ============================================================

rows = []

for paper in all_papers:
    authors = []
    institutions = []
    countries = []

    for auth in safe_list(paper.get("authorships")):
        author_name = safe_get(auth, ["author", "display_name"])
        if author_name:
            authors.append(author_name)

        for c in safe_list(auth.get("countries")):
            countries.append(c)

        for inst in safe_list(auth.get("institutions")):
            inst_name = inst.get("display_name")
            inst_country = inst.get("country_code")

            if inst_name:
                institutions.append(inst_name)

            if inst_country:
                countries.append(inst_country)

    topics = []
    for topic in safe_list(paper.get("topics")):
        topic_name = topic.get("display_name")
        if topic_name:
            topics.append(topic_name)

    publication_year = paper.get("publication_year")
    citation_count = paper.get("cited_by_count") or 0

    if publication_year:
        paper_age = max(1, CURRENT_YEAR - int(publication_year) + 1)
        citations_per_year = citation_count / paper_age
    else:
        paper_age = None
        citations_per_year = None

    row = {
        # Publication metadata
        "paper_id": paper.get("id"),
        "title": paper.get("title") or paper.get("display_name"),
        "publication_year": publication_year,
        "publication_type": paper.get("type"),

        # Impact metrics
        "citation_count": citation_count,
        "citations_per_year": citations_per_year,
        "referenced_works_count": paper.get("referenced_works_count"),

        # Research context
        "topics": join_unique(topics),
        "primary_topic": safe_get(paper, ["primary_topic", "display_name"]),
        "primary_subfield": safe_get(paper, ["primary_topic", "subfield", "display_name"]),
        "primary_field": safe_get(paper, ["primary_topic", "field", "display_name"]),
        "venue_source": safe_get(paper, ["primary_location", "source", "display_name"]),

        # Contributor metadata
        "authors": join_unique(authors),
        "institutions": join_unique(institutions),
        "countries": join_unique(countries),

        # Useful extra filters
        "doi": paper.get("doi"),
        "language": paper.get("language"),
        "is_oa": safe_get(paper, ["open_access", "is_oa"]),
        "oa_status": safe_get(paper, ["open_access", "oa_status"]),
    }

    rows.append(row)


df = pd.DataFrame(rows)

df = df.drop_duplicates(subset=["paper_id"])

df = df.sort_values(
    by=["publication_year", "citation_count"],
    ascending=[True, False]
)

df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")

print(f"Saved clean CSV to {CSV_FILE}")
print(df.head())
print(df.shape)

Starting data download from OpenAlex...
Downloaded usable papers: 155 / 10000
Downloaded usable papers: 305 / 10000
Downloaded usable papers: 456 / 10000
Downloaded usable papers: 603 / 10000
Downloaded usable papers: 746 / 10000
Downloaded usable papers: 895 / 10000
Downloaded usable papers: 1032 / 10000


In [ ]:
import pandas as pd

df = pd.read_csv(CSV_FILE)

print(df.shape)
print(df.info())

print("\nMissing values:")
print(df.isna().mean().sort_values(ascending=False))

print("\nPublication years:")
print(df["publication_year"].describe())

print("\nPublication types:")
print(df["publication_type"].value_counts())

print("\nTop fields:")
print(df["primary_field"].value_counts().head(10))

print("\nTop countries:")
print(df["countries"].value_counts().head(10))

print("\nCitation summary:")
print(df[["citation_count", "citations_per_year", "referenced_works_count"]].describe())